
# Business Challenge

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

(Thanks to Edward Donner for the lesson!)

---
## Setup

In [1]:
# Imports

import json
from IPython.display import Markdown, display, update_display

from openai import OpenAI
MODEL = "llama3.2"

from bs4 import BeautifulSoup
import requests

In [ ]:
# Connect to the Ollama API
llama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

---

## Part 1 - Relevant Links' Extraction

### Scraper and Link Extractor

In [3]:
# Website Scrapers 

# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

### Define Prompts and Prompt Generators

In [12]:
# system prompt for the links' extracton task
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to an About page, or a Company page, or Careers/Jobs pages.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    }
"""

In [13]:
# user prompt for the links' extraction task

def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    """
    
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
# run the above function with input to check out the resulting user prompt
print(get_links_user_prompt("https://www.apple.com/"))


        Here is the list of links on the website https://www.apple.com/ -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    /
/us/shop/goto/store
/mac/
/ipad/
/iphone/
/watch/
/apple-vision-pro/
/airpods/
/tv-home/
/entertainment/
/us/shop/goto/buy_accessories
https://support.apple.com/?cid=gn-ols-home-hp-tab
/us/search
/us/shop/goto/bag
/iphone-18-pro/
/iphone-18-pro/
/us/shop/goto/buy_iphone/iphone_18_pro
/iphone-duo/
/iphone-duo/
/us/shop/goto/buy_iphone/iphone_duo
/apple-watch-series-12/
/apple-watch-series-12/
/us/shop/goto/buy_watch/apple_watch_series_12
/apple-watch-ultra-4/
/apple-watch-ultra-4/
/us/shop/goto/buy_watch/apple_watch_ultra_4
/macbook-air/
/macbook-air/
/us/shop/goto/buy_mac/macbook_air
/ipad-air/
/ipad-air/
/us/shop/goto/buy_ipad/ipad_air
/airpods-pro

### Link Extraction with Llama3.2

In [18]:
# link extraction function that uses the above system and user prompts to extract relevant links from a website
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = llama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content # JSON string from the model
    links = json.loads(result) # JSON string to python dictionary

    print(f"Found {len(links['links'])} relevant links")
    print(links)
    return links

In [11]:
select_relevant_links("https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 35 relevant links


{'links': [{'type': 'Company page', 'url': 'https://www.apple.com'},
  {'type': 'About page',
   'url': 'https://www.apple.com/investor relations/default.aspx'},
  {'type': 'Careers page', 'url': 'https://careers.apple.com/'},
  {'type': 'Newsroom', 'url': 'https://www.apple.com/newsroom/'},
  {'type': 'Investor Relations', 'url': 'https://investor.apple.com/'},
  {'type': 'Terms of Service', 'url': 'https://www.apple.com/terms.html'},
  {'type': 'FAQs', 'url': 'https://www.apple.com/FAQ/'},
  {'type': 'Contact', 'url': 'https://www.apple.com/contact/'},
  {'type': 'Support', 'url': 'https://support.apple.com/'},
  {'type': 'Apple Card', 'url': 'https://card.apple.com/apply/application'},
  {'type': 'Apple Wallet', 'url': 'https://applepay.apple.com/'},
  {'type': 'Apple Cash',
   'url': 'https://apple-cash.apple.com/redirect?uri=https://apple-cash.apple.com/'},
  {'type': 'Apple Pay', 'url': 'https://applepay.apple.com/'},
  {'type': 'Services', 'url': 'https://www.apple.com/services/

--- 
## Part 2 - Create the Brochure

In [29]:
# function to fetch the main page content and all relevant links from a website
def fetch_content_of_page_and_for_all_relevant_links(url):
    
    # fetch the main page content
    contents = fetch_website_contents(url)
    
    # fetch the relevant links with the LLM
    relevant_links = select_relevant_links(url)
    
    # format the result as a markdown string
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    # fetch the contents of each relevant link and append to the result
    for link in relevant_links['links']:

        try:
            # fetch the contents of the relevant link
            link_contents = fetch_website_contents(link["url"])

            result += f"\n\n### Link: {link['type']}\n"
            result += link_contents
        except Exception as e:
            print(f"Error fetching contents for {link['url']}: {e}")
            # link_contents = "Error fetching contents."


    return result

In [30]:
# run the content fetching function for a website and display the result as markdown
result = fetch_content_of_page_and_for_all_relevant_links("https://www.apple.com")

print(result)

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 13 relevant links
{'links': ['https://www.apple.com/', 'https://card.apple.com/apply/application?referrer=cid%3Dapy-200-10000036&start=false', 'https://account.apple.com/', 'https://apple-one.apple.com/', 'https://www.icloud.com/', 'https://apps.apple.com/us/app/apple-store/id375380948', 'https://apple.com/careers', 'https://investor.apple.com/', 'https://newsroom.apple.com/', 'https://locate.apple.com/', 'https://support.apple.com/', 'https://www.apple.com/shop', 'https://www.apple.com/airtag']}


TypeError: string indices must be integers, not 'str'

### Define Prompts and Prompt Generator

In [23]:
# system prompt for brochure generation task

brochure_system_prompt = """
    You are an assistant that analyzes the contents of several relevant pages from a company website
    and creates a short brochure about the company for prospective customers, investors and recruits.
    Respond in markdown without code blocks.
    Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
#   You are an assistant that analyzes the contents of several relevant pages from a company website
#   and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
#   Respond in markdown without code blocks.
#   Include details of company culture, customers and careers/jobs if you have the information.
#   """

In [26]:
# user prompt generator function
def get_brochure_user_prompt(company_name, url):
    
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """

    user_prompt += fetch_content_of_page_and_for_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    
    return user_prompt

In [27]:
# Call the brochure generation user prompt function
get_brochure_user_prompt("Apple", "https://www.apple.com")

Selecting relevant links for https://www.apple.com by calling llama3.2
Found 6 relevant links
{'links': ['https://www.apple.com/about/', 'https://www.apple.com/careers/us/', 'https://investor.apple.com', 'https://about.apple.com/press-room', 'https://www.apple.com/newsroom', 'https://locate.apple.com']}


TypeError: string indices must be integers, not 'str'